In [0]:
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import DoubleType, LongType

# Pegar todos os parquets
file_list = dbutils.fs.ls("s3://trip-records-taxi-data/yellow_taxi/")
parquet_files = [f.path for f in file_list if f.path.endswith('.parquet')]

# Ler o primeiro parquet
df_combined = spark.read.parquet(parquet_files[0])

# Passar para o tipo de dados correto
df_combined = df_combined \
    .withColumn("VendorID", col("VendorID").cast(LongType())) \
    .withColumn("passenger_count", col("passenger_count").cast(DoubleType())) \
    .withColumn("RatecodeID", col("RatecodeID").cast(DoubleType())) \
    .withColumn("PULocationID", col("PULocationID").cast(LongType())) \
    .withColumn("DOLocationID", col("DOLocationID").cast(LongType()))

# Normalizando o nome das coluna
for column in df_combined.columns:
    if column.lower() == "airport_fee":
        df_combined = df_combined.withColumnRenamed(column, "airport_fee")

# Juntando os demais parquets
for file_path in parquet_files[1:]:
    df_temp = spark.read.parquet(file_path)
    
    # Passar para o tipo de dados correto
    df_temp = df_temp \
        .withColumn("VendorID", col("VendorID").cast(LongType())) \
        .withColumn("passenger_count", col("passenger_count").cast(DoubleType())) \
        .withColumn("RatecodeID", col("RatecodeID").cast(DoubleType())) \
        .withColumn("PULocationID", col("PULocationID").cast(LongType())) \
        .withColumn("DOLocationID", col("DOLocationID").cast(LongType()))
    
    # Normalizando o nome das coluna
    for column in df_temp.columns:
        if column.lower() == "airport_fee":
            df_temp = df_temp.withColumnRenamed(column, "airport_fee")
    
    df_combined = df_combined.unionByName(df_temp, allowMissingColumns=True)

# Add timestamp da ingestão
df_combined = df_combined.withColumn("ingestion_timestamp", current_timestamp())

df_combined.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.taxidata.yellow_taxi_data")

In [0]:
df1 = spark.table("bronze.taxidata.yellow_taxi_data")

In [0]:
df1.count()

In [0]:
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import DoubleType, LongType

# Pegar todos os parquets
file_list = dbutils.fs.ls("s3://trip-records-taxi-data/green_taxi/")
parquet_files = [f.path for f in file_list if f.path.endswith('.parquet')]

# Ler o primeiro parquet
df_combined = spark.read.parquet(parquet_files[0])

# Passar para o tipo de dados correto
df_combined = df_combined \
    .withColumn("VendorID", col("VendorID").cast(LongType())) \
    .withColumn("passenger_count", col("passenger_count").cast(DoubleType())) \
    .withColumn("RatecodeID", col("RatecodeID").cast(DoubleType())) \
    .withColumn("PULocationID", col("PULocationID").cast(LongType())) \
    .withColumn("DOLocationID", col("DOLocationID").cast(LongType()))

# Juntando os demais parquets
for file_path in parquet_files[1:]:
    df_temp = spark.read.parquet(file_path)
    
    # Passar para o tipo de dados correto
    df_temp = df_temp \
        .withColumn("VendorID", col("VendorID").cast(LongType())) \
        .withColumn("passenger_count", col("passenger_count").cast(DoubleType())) \
        .withColumn("RatecodeID", col("RatecodeID").cast(DoubleType())) \
        .withColumn("PULocationID", col("PULocationID").cast(LongType())) \
        .withColumn("DOLocationID", col("DOLocationID").cast(LongType()))
    
    df_combined = df_combined.unionByName(df_temp, allowMissingColumns=True)

# Add timestamp da ingestão
df_combined = df_combined.withColumn("ingestion_timestamp", current_timestamp())

df_combined.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.taxidata.green_taxi_data")

In [0]:
df2 = spark.table("bronze.taxidata.green_taxi_data")

In [0]:
df2.printSchema()